# Initialization

## Libraries

In [1]:
%load_ext autoreload
%autoreload 2

from skopt import gp_minimize
from skopt.learning import GaussianProcessRegressor
from skopt.learning.gaussian_process.kernels import Matern, RBF
from skopt.space import Real, Categorical

from scipy.spatial.distance import cdist
from sklearn.preprocessing import MinMaxScaler

import warnings
import numpy as np
import pandas as pd
from pathlib import Path

## Directories

In [2]:
data_dir = Path("data")
Path.mkdir(data_dir, exist_ok=True)

plot_dir = Path("plots")
Path.mkdir(plot_dir, exist_ok=True)

log_dir = Path("logs")
Path.mkdir(log_dir, exist_ok=True)

## Data reading

In [3]:
data_file = "Dataset_SL.xlsx"

### Define experimental data

In [4]:
# 1. Define the experimental data
experiment_data = pd.read_excel(data_dir / data_file, sheet_name="Datasheet")

In [5]:
# make the other columns as floats
experiment_data = experiment_data.astype(
    {
        "salt_concentration": float,
        "water_to_cement_ratio": float,
        "antisettling_concentration": float,
        "E_d": float,
        "KPI": float,
    }
)

# get optimization round for later use in saving
opt_round = experiment_data["opt_round"].iloc[-1]

In [6]:
experiment_data

,opt_round,salt,salt_concentration,water_to_cement_ratio,antisettling_concentration,kernel,acquisition,obj_func,n_rejected_before_accept,E_d,KPI,Notes
0,0,MgSO4,0.50,1.00,0.00,NaN,NaN,NaN,NaN,39.392651,6.764730,NaN
1,0,CaCl2,0.50,1.00,0.00,NaN,NaN,NaN,NaN,91.053468,2.427368,NaN
2,0,SrBr2,0.50,1.00,0.00,NaN,NaN,NaN,NaN,159.169079,32.224874,NaN
3,0,MgCl2,0.50,1.00,0.00,NaN,NaN,NaN,NaN,88.671748,2.715363,NaN
4,0,CuSO4,0.50,1.00,0.00,NaN,NaN,NaN,NaN,9.658742,61.776688,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
77,3,Zn(NO3)2,0.48,1.34,0.00,RBF,EI,KPI,27.0,105.634319,9.707920,NaN
78,3,MgCl2,0.39,1.04,1.21,RBF,PI,KPI,0.0,62.079727,6.429156,NaN
79,3,Mg(NO3)2,0.21,0.91,0.11,RBF,LCB_low,KPI,33.0,40.210829,9.121303,NaN
80,3,KAl(SO4)2,0.73,1.40,0.00,RBF,LCB_med,KPI,34.0,23.236032,8.052443,NaN


# Search space

### Category encoding

In [7]:
cat_order = [
    "Al2(SO4)3",
    "CaCl2",
    "CuSO4",
    "K2CO3",
    "KAl(SO4)2",
    "LiCl",
    "Mg(NO3)2",
    "MgCl2",
    "MgSO4",
    "SrBr2",
    "Zn(NO3)2",
]

cat_mapping = {name: i for i, name in enumerate(cat_order)}

df_enc = experiment_data.copy()
df_enc["category_encoded"] = experiment_data["salt"].map(cat_mapping)
df_enc

,opt_round,salt,salt_concentration,water_to_cement_ratio,antisettling_concentration,kernel,acquisition,obj_func,n_rejected_before_accept,E_d,KPI,Notes,category_encoded
0,0,MgSO4,0.50,1.00,0.00,NaN,NaN,NaN,NaN,39.392651,6.764730,NaN,8
1,0,CaCl2,0.50,1.00,0.00,NaN,NaN,NaN,NaN,91.053468,2.427368,NaN,1
2,0,SrBr2,0.50,1.00,0.00,NaN,NaN,NaN,NaN,159.169079,32.224874,NaN,9
3,0,MgCl2,0.50,1.00,0.00,NaN,NaN,NaN,NaN,88.671748,2.715363,NaN,7
4,0,CuSO4,0.50,1.00,0.00,NaN,NaN,NaN,NaN,9.658742,61.776688,NaN,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,3,Zn(NO3)2,0.48,1.34,0.00,RBF,EI,KPI,27.0,105.634319,9.707920,NaN,10
78,3,MgCl2,0.39,1.04,1.21,RBF,PI,KPI,0.0,62.079727,6.429156,NaN,7
79,3,Mg(NO3)2,0.21,0.91,0.11,RBF,LCB_low,KPI,33.0,40.210829,9.121303,NaN,6
80,3,KAl(SO4)2,0.73,1.40,0.00,RBF,LCB_med,KPI,34.0,23.236032,8.052443,NaN,4


In [8]:
search_space = [
    Categorical(cat_order, name="salt"),  # salt names
    Real(0.1, 0.9, name="salt_concentration"),
    Real(0.7, 1.5, name="water_to_cement_ratio"),
    Real(0.0, 3.0, name="antisettling_concentration"),
]

# Optimization

In [9]:
salt_dummies = pd.get_dummies(df_enc["salt"], prefix="salt").reindex(
    columns=[f"salt_{cat}" for cat in cat_order], fill_value=0
)

In [10]:
X = pd.concat(
    [
        salt_dummies,
        df_enc[
            [
                "salt_concentration",
                "water_to_cement_ratio",
                "antisettling_concentration",
            ]
        ],
    ],
    axis=1,
).to_numpy()
y_energy = df_enc["E_d"].values
y_kpi = df_enc["KPI"].values

In [11]:
X

array([[False, False, False, ..., 0.5, 1.0, 0.0],
       [False, True, False, ..., 0.5, 1.0, 0.0],
       [False, False, False, ..., 0.5, 1.0, 0.0],
       ...,
       [False, False, False, ..., 0.21, 0.91, 0.11],
       [False, False, False, ..., 0.73, 1.4, 0.0],
       [False, False, False, ..., 0.34, 0.87, 0.0]],
      shape=(82, 14), dtype=object)

## Gaussian Process

In [12]:
def fit_gp_models(X, y, kernel):
    """Fits Gaussian Process models to the given experimental data."""  # noqa

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        alpha=1e-3,
    )

    gp.fit(X, y)

    return gp

### Matern kernel

In [13]:
matern_kernel = Matern(
    length_scale=5e-3, length_scale_bounds=(1e-8, 10.0), nu=2.5
)  # noqa

In [14]:
gp_energy_matern = fit_gp_models(X, -y_energy, matern_kernel)

In [15]:
gp_kpi_matern = fit_gp_models(X, y_kpi, matern_kernel)

### RBF kernel

In [16]:
rbf_kernel = RBF(length_scale=5e-3, length_scale_bounds=(1e-8, 10.0))

In [17]:
gp_energy_rbf = fit_gp_models(X, -y_energy, rbf_kernel)

In [18]:
gp_kpi_rbf = fit_gp_models(X, y_kpi, rbf_kernel)

# Bayesian Optimization

In [19]:
def encode_input(x):
    # x[0] is salt name (e.g., "MgSO4")
    salt_vector = np.zeros(len(cat_mapping))
    salt_index = cat_mapping[x[0]]
    salt_vector[salt_index] = 1

    # concatenate with continuous features
    return np.concatenate([salt_vector, np.array(x[1:])])

In [20]:
def suggest_new_samples_with_filtering(
    gp_model: GaussianProcessRegressor,
    kernel: Matern | RBF,
    obj_func: str,
    existing_X: np.ndarray,
    distance_threshold: float = 0.05,
    print_points: bool = False,
    verbose: bool = False,
    mute_warnings: bool = True,
):
    acq_functions = ["EI", "PI", "LCB_low", "LCB_med", "LCB_high"]
    kappa_values = {
        "LCB_low": 1.0,
        "LCB_med": 4.0,
        "LCB_high": 10.0,
    }

    # Scale existing data once for consistent distance computation
    scaler = MinMaxScaler()
    X_scaled_existing = scaler.fit_transform(existing_X)

    new_samples = []

    with warnings.catch_warnings():
        if mute_warnings:
            warnings.simplefilter("ignore")

        for acquisition in acq_functions:
            res = gp_minimize(
                lambda x: gp_model.predict([encode_input(x)])[0],
                dimensions=search_space,
                base_estimator=GaussianProcessRegressor(
                    kernel=kernel,
                    normalize_y=True,
                    n_restarts_optimizer=10,
                    alpha=1e-3,
                ),
                acq_func=("LCB" if "LCB" in acquisition else acquisition),
                kappa=kappa_values.get(acquisition),
                xi=0.1,
                n_calls=50,
                verbose=verbose,
                n_jobs=6,
            )

            # Sort all candidates by predicted value
            scored_candidates = sorted(
                zip(res.x_iters, res.func_vals), key=lambda x: x[1]
            )

            accepted_count = 0
            rejected_total = 0

            for candidate_x, _ in scored_candidates:
                x_enc = encode_input(candidate_x).reshape(1, -1)
                x_scaled = scaler.transform(x_enc)
                min_dist = cdist(x_scaled, X_scaled_existing).min()

                if min_dist >= distance_threshold:
                    candidate_x.append(str(gp_model.kernel).split("(")[0])
                    candidate_x.append(str(acquisition))
                    candidate_x.append(str(obj_func))
                    candidate_x.append(rejected_total)

                    new_samples.append(candidate_x)

                    X_scaled_existing = np.vstack([X_scaled_existing, x_scaled])
                    accepted_count += 1
                    break  # accept only one point per acquisition
                else:
                    rejected_total += 1

            if print_points:
                print(
                    f"{acquisition}: Accepted {accepted_count} after {rejected_total} rejections"
                )

            # Fallback if nothing accepted
            if accepted_count == 0:
                fallback_x, _ = scored_candidates[0]
                fallback_x.append(str(gp_model.kernel).split("(")[0])
                fallback_x.append(str(acquisition))
                fallback_x.append(str(obj_func))
                fallback_x.append(rejected_total)
                new_samples.append(fallback_x)

                if print_points:
                    print(
                        f"{acquisition}: Fallback used after {rejected_total} rejections"
                    )

    df = pd.DataFrame(
        new_samples,
        columns=[
            "salt",
            "salt_concentration",
            "water_to_cement_ratio",
            "antisettling_concentration",
            "kernel",
            "acquisition",
            "obj_func",
            "n_rejected_before_accept",
        ],
    )

    df[df.select_dtypes(include="float").columns] = df.select_dtypes(
        include="float"
    ).round(2)

    return df

### Generate 10 new samples for Energy Density optimization

In [21]:
# Matérn kernel
new_samples_energy_matern = suggest_new_samples_with_filtering(
    gp_energy_matern,
    matern_kernel,
    obj_func="E_d",
    existing_X=X,
    distance_threshold=0.2,  # adjust as needed
    print_points=True,
    verbose=True,
)

Iteration No: 1 started. Evaluating function at random point.
Iteration No: 1 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: -39.6215
Current minimum: -39.6215
Iteration No: 2 started. Evaluating function at random point.
Iteration No: 2 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: -84.4669
Current minimum: -84.4669
Iteration No: 3 started. Evaluating function at random point.
Iteration No: 3 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: -78.3162
Current minimum: -84.4669
Iteration No: 4 started. Evaluating function at random point.
Iteration No: 4 ended. Evaluation done at random point.
Time taken: 0.0000
Function value obtained: -73.0723
Current minimum: -84.4669
Iteration No: 5 started. Evaluating function at random point.
Iteration No: 5 ended. Evaluation done at random point.
Time taken: 0.0061
Function value obtained: -88.0714
Current minimum: -88.0714
Iteration No: 6

In [22]:
new_samples_energy_matern

,salt,salt_concentration,water_to_cement_ratio,antisettling_concentration,kernel,acquisition,obj_func,n_rejected_before_accept
0,LiCl,0.79,1.37,0.29,Matern,EI,E_d,10
1,CaCl2,0.59,1.32,2.19,Matern,PI,E_d,6
2,CaCl2,0.90,1.43,3.00,Matern,LCB_low,E_d,37
3,Zn(NO3)2,0.17,1.50,0.00,Matern,LCB_med,E_d,0
4,LiCl,0.90,1.50,0.00,Matern,LCB_high,E_d,7


In [23]:
# RBF kernel
new_samples_energy_rbf = suggest_new_samples_with_filtering(
    gp_energy_rbf,
    rbf_kernel,
    obj_func="E_d",
    existing_X=X,
    distance_threshold=0.2,  # adjust as needed
    print_points=True,
)

EI: Accepted 1 after 0 rejections
PI: Accepted 1 after 0 rejections
LCB_low: Accepted 1 after 23 rejections
LCB_med: Accepted 1 after 9 rejections
LCB_high: Accepted 1 after 0 rejections


In [24]:
new_samples_energy_rbf

,salt,salt_concentration,water_to_cement_ratio,antisettling_concentration,kernel,acquisition,obj_func,n_rejected_before_accept
0,LiCl,0.76,1.49,0.10,RBF,EI,E_d,0
1,MgCl2,0.57,1.09,0.99,RBF,PI,E_d,0
2,CaCl2,0.90,1.41,3.00,RBF,LCB_low,E_d,23
3,LiCl,0.44,1.33,0.00,RBF,LCB_med,E_d,9
4,LiCl,0.10,1.23,0.00,RBF,LCB_high,E_d,0


### Generate 10 new samples for economic KPI optimization

In [25]:
# Matérn kernel
new_samples_kpi_matern = suggest_new_samples_with_filtering(
    gp_kpi_matern,
    matern_kernel,
    obj_func="KPI",
    existing_X=X,
    distance_threshold=0.2,  # adjust as needed
    print_points=True,
)

EI: Accepted 1 after 17 rejections
PI: Accepted 1 after 21 rejections
LCB_low: Accepted 1 after 37 rejections
LCB_med: Accepted 1 after 27 rejections
LCB_high: Accepted 1 after 14 rejections


In [26]:
new_samples_kpi_matern

,salt,salt_concentration,water_to_cement_ratio,antisettling_concentration,kernel,acquisition,obj_func,n_rejected_before_accept
0,CaCl2,0.51,1.25,2.06,Matern,EI,KPI,17
1,Zn(NO3)2,0.64,1.18,0.17,Matern,PI,KPI,21
2,CaCl2,0.58,1.41,1.91,Matern,LCB_low,KPI,37
3,KAl(SO4)2,0.84,1.15,0.00,Matern,LCB_med,KPI,27
4,Zn(NO3)2,0.10,1.23,1.04,Matern,LCB_high,KPI,14


In [27]:
# RBF kernel
new_samples_kpi_rbf = suggest_new_samples_with_filtering(
    gp_kpi_rbf,
    rbf_kernel,
    obj_func="KPI",
    existing_X=X,
    distance_threshold=0.2,  # adjust as needed
    print_points=True,
)

EI: Accepted 1 after 15 rejections
PI: Accepted 1 after 34 rejections
LCB_low: Accepted 1 after 8 rejections
LCB_med: Accepted 1 after 4 rejections
LCB_high: Accepted 1 after 12 rejections


In [28]:
new_samples_kpi_rbf

,salt,salt_concentration,water_to_cement_ratio,antisettling_concentration,kernel,acquisition,obj_func,n_rejected_before_accept
0,MgCl2,0.15,1.25,0.00,RBF,EI,KPI,15
1,MgSO4,0.33,0.88,0.01,RBF,PI,KPI,34
2,MgCl2,0.55,1.07,1.07,RBF,LCB_low,KPI,8
3,Zn(NO3)2,0.16,1.24,1.04,RBF,LCB_med,KPI,4
4,Zn(NO3)2,0.65,1.40,0.00,RBF,LCB_high,KPI,12


## Saving the new suggestions in the original excel sheet

In [29]:
# Combine sets of new samples
new_samples = pd.concat(
    [
        new_samples_energy_matern,
        new_samples_energy_rbf,
        new_samples_kpi_matern,
        new_samples_kpi_rbf,
    ],
    ignore_index=True,
)

new_samples.insert(0, "opt_round", opt_round + 1)  # add optimization round

# Read the existing Excel sheet into a DataFrame
experiment_data = pd.read_excel(data_dir / data_file, sheet_name="Datasheet")
# Append the new data to the existing DataFrame
combined_data = pd.concat(
    [experiment_data, new_samples], ignore_index=True, axis=0
)  # noqa

# Write the updated DataFrame back to the same Excel sheet
with pd.ExcelWriter(
    data_dir / data_file, engine="openpyxl", mode="a", if_sheet_exists="replace"  # noqa
) as writer:
    combined_data.to_excel(writer, sheet_name="Datasheet", index=False)

print(
    "New batch of 20 samples suggested. Please conduct experiments and update the dataset."  # noqa
)

New batch of 20 samples suggested. Please conduct experiments and update the dataset.
